In [11]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
# pdf_directory = r"E:\learn-rag\data\arxiv"

In [15]:
# from pathlib import Path
# from langchain_community.document_loaders import PyPDFLoader

# def process_all_pdfs(pdf_directory):
#     all_documents = []
#     pdf_dir = Path(pdf_directory)
#     pdf_files = list(pdf_dir.glob("**/*.pdf"))
#     print(f"Found {len(pdf_files)} PDF files")
#     for pdf_file in pdf_files:
#         print(f"Processing: {pdf_file.name}")
#         loader = PyPDFLoader(str(pdf_file))
#         documents = loader.load()
#         all_documents.extend(documents)
#     return all_documents

# pdf_directory = r"E:\learn-rag\data\arxiv"
# documents = process_all_pdfs(pdf_directory)
# print(len(documents))

In [16]:
# all_documents.extend(documents)

In [18]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

# Folder containing PDFs
pdf_folder = r"E:\learn-rag\data\arxiv"

# Find all PDF files
pdf_files = list(Path(pdf_folder).glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files\n")

# Store all loaded pages
all_documents = []

# Load each PDF
for pdf in pdf_files:

    print(f"Loading: {pdf.name}")

    loader = PyPDFLoader(str(pdf))

    documents = loader.load()

    all_documents.extend(documents)

print(f"\nTotal pages loaded: {len(all_documents)}")

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 35 0 (offset 0)
Ignoring wrong pointing object 37 0 (offset 0)
Ignoring wrong pointing object 44 0 (offset 0)
Ignoring wrong pointing object 53 0 (offset 0)
Ignoring wrong pointing object 80 0 (offset 0)
Ignoring wrong pointing object 85 0 (offset 0)
Ignoring wrong pointing object 87 0 (offset 0)
Ignoring wrong pointing object 91 0 (offset 0)
Ignoring wrong pointing object 93 0 (offset 0)
Ignoring wrong pointing object 109 0 (offset 0)


Found 8 PDF files

Loading: An on-chip photonic deep neural network for image classification.pdf
Loading: Attojoule optoelectronics for communications and computing.pdf
Loading: Deep learning with coherent nanophotonic circuits.pdf
Loading: Experimentally realized in situ backpropagation.pdf
Loading: Optimal design for universal multiport interferometers.pdf
Loading: Parallel convolutional processing using an integrated photonic tensor core.pdf
Loading: Photonic matrix multiplication lights up photonic accelerator and beyond.pdf
Loading: Photonic softmax jstqe submit.pdf

Total pages loaded: 172


In [19]:
print(all_documents[0].page_content[:500])

1 
Single-chip photonic deep neural network for instantaneous image classification  Farshid Ashtiani, Alexander J. Geers, and Firooz Aflatouni*  Department of Electrical and Systems Engineering, University of Pennsylvania, Philadelphia, PA 19104, USA *firooz@seas.upenn.edu Deep neural networks with applications from computer vision and image processing to medical diagnosis1-5 are commonly implemented using clock-based processors6-14, where computation speed is generally limited by the clock freq


In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# Split documents into chunks
chunks = splitter.split_documents(all_documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 970


In [22]:
print(chunks[10].page_content)

neural network classifier implemented in Python environment using Keras31 achieves 96% accuracy for the same data set. The implemented PDNN features direct clock-less processing of input images which eliminates the need for photo-detection, scaling and amplification, analogue-to-digital conversion, data alignment, and a large memory module enabling the realization of significantly faster and more energy-efficient, yet privacy-aware neural networks for the next generations of deep learning systems. The PDNN chip was integrated within a footprint of 9.3 mm2.


In [23]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\Atharva\anaconda3\envs\rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4315.90it/s]


In [24]:
vector = embeddings.embed_query(chunks[0].page_content)

print(len(vector))

384


In [25]:
from langchain_chroma import Chroma

# Create vector database
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("ChromaDB created successfully")

ChromaDB created successfully


In [26]:
results = vectorstore.similarity_search(
    "What is photonic matrix multiplication?",
    k=3
)

print(results[0].page_content)

multiplication, to address the growing demand for computing resources and capacity. Photonic matrix multiplication
has much potential to expand the domain of telecommunication, and arti ﬁcial intelligence bene ﬁting from its superior
performance. Recent research in photonic matrix multiplication has ﬂourished and may provide opportunities to
develop applications that are unachievable at present by conventional electronic processors. In this review, we ﬁrst
introduce the methods of photonic matrix multiplication, mainly including the plane light conversion method,
Mach–Zehnder interferometer method and wavelength division multiplexing method. We also summarize the
developmental milestones of photonic matrix multiplication and the related applications. Then, we review their
detailed advances in applications to optical signal processing and arti ﬁcial neural networks in recent years. Finally, we


In [27]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

docs = retriever.invoke(
    "What is photonic matrix multiplication?"
)

print(docs[0].page_content)

multiplication, to address the growing demand for computing resources and capacity. Photonic matrix multiplication
has much potential to expand the domain of telecommunication, and arti ﬁcial intelligence bene ﬁting from its superior
performance. Recent research in photonic matrix multiplication has ﬂourished and may provide opportunities to
develop applications that are unachievable at present by conventional electronic processors. In this review, we ﬁrst
introduce the methods of photonic matrix multiplication, mainly including the plane light conversion method,
Mach–Zehnder interferometer method and wavelength division multiplexing method. We also summarize the
developmental milestones of photonic matrix multiplication and the related applications. Then, we review their
detailed advances in applications to optical signal processing and arti ﬁcial neural networks in recent years. Finally, we


In [ ]:
os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0
)

In [ ]:
query = "What is photonic matrix multiplication?"

docs = retriever.invoke(query)

In [ ]:
context = "\n\n".join([doc.page_content for doc in docs])

In [ ]:
prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print(response.content)